# Camada Silver

In [0]:
-- Define o catálogo e o schema onde a tabela Silver será criada

USE CATALOG mvp;
USE SCHEMA silver;

In [0]:
-- Cria (ou substitui) a tabela da camada Silver
CREATE OR REPLACE TABLE producao_energetica AS

-- Padronização dos campos textuais
-- Remove espaços em branco antes e depois dos valores.
-- Remove coluna Regiao
SELECT
  trim(Unidades_da_Federacao) AS uf,

-- Renomeia a coluna para deixar explícito que representa a unidade de medida da produção (m³, mil barris, etc.)
  trim(Unidade) AS unidade_medida,
  trim(Produto) AS produto,

-- Conversão do ano para o tipo inteiro
  CAST(Ano AS INT) as ano,


-- Substitui "-" por NULL;
-- Remove separador de milhares (.);
-- Converte vírgula decimal para ponto;
-- Converte o valor para DOUBLE.
  CASE
    WHEN TRIM(Producao) = '-' THEN NULL
    ELSE CAST(REPLACE(REPLACE(Producao, '.', ''), ',', '.') AS DOUBLE)
  END AS producao

FROM (

    SELECT
        Unidades_da_Federacao,
        Unidade,
        Produto,
    
-- Transformação do formato Wide para Long
        STACK(
            10,

            2014, `2014`,
            2015, `2015`,
            2016, `2016`,
            2017, `2017`,
            2018, `2018`,
            2019, `2019`,
            2020, `2020`,
            2021, `2021`,
            2022, `2022`,
            2023, `2023`
        ) AS (Ano, Producao)

    FROM mvp.bronze.producao_energetica
);

num_affected_rows,num_inserted_rows


In [0]:
-- Consulta a tabela Silver para validar o resultado
-- das etapas de limpeza, padronização e transformação dos dados.

SELECT *
FROM mvp.silver.producao_energetica

uf,unidade_medida,produto,ano,producao
Amazonas,(mil barris),LGN,2014,6085.0
Ceara,(mil barris),LGN,2014,57.0
Rio Grande do Norte,(mil barris),LGN,2014,1338.0
Alagoas,(mil barris),LGN,2014,516.0
Sergipe,(mil barris),LGN,2014,1084.0
Bahia,(mil barris),LGN,2014,1484.0
Espirito Santo,(mil barris),LGN,2014,6140.0
Rio de Janeiro,(mil barris),LGN,2014,15177.0
Sao Paulo,(mil barris),LGN,2014,1594.0
Amazonas,(mil barris),LGN,2015,6366.0


### 2. Catálogo de Dados

| Coluna             | Tipo   | Descrição                                                                                                                                                                                                                        |
| ------------------ | ------ | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **uf**             | string | Unidade da Federação responsável pela produção energética. Corresponde ao estado brasileiro onde a produção foi registrada.                                                                                                      |
| **unidade_medida** | string | Unidade de medida utilizada para representar a produção do produto energético (ex.: m³, mil m³, mil barris).                                                                                                                     |
| **produto**        | string | Produto energético produzido, conforme classificação da ANP (ex.: Petróleo, Gás Natural, Biodiesel, Biometano, Etanol Anidro e Hidratado, LGN).                                                                                  |
| **ano**            | int    | Ano de referência da produção energética. Valores compreendidos entre 2014 e 2023.                                                                                                                                               |
| **producao**       | double | Quantidade produzida do respectivo produto energético na Unidade da Federação, expressa na unidade de medida correspondente. Valores ausentes foram convertidos para **NULL** durante o processo de tratamento da camada Silver. |
